In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Fraud Detection AI")
    .getOrCreate()
)

# Load dataset
df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("../data/raw/creditcard.csv")
)

In [2]:
#Print Schema
df.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [3]:
# Dataset dimensions
num_rows = df.count()
num_columns = len(df.columns)

print(f"Rows: {num_rows:,}")
print(f"Columns: {num_columns}")

Rows: 284,807
Columns: 31


In [4]:
# ==========================
# Missing Values Analysis
# ==========================

from pyspark.sql.functions import col, when, sum

df.select(
    [
        sum(
            when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]
).show()

+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|Time| V1| V2| V3| V4| V5| V6| V7| V8| V9|V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|Class|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|   0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|     0|    0|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+



In [5]:
# ==========================================================
# DISTINCT - Compare Original vs Distinct DataFrame
# ==========================================================

# Remove duplicate rows (Transformation)
df_distinct = df.distinct()

# Compare row counts
original_rows = df.count()
distinct_rows = df_distinct.count()

print(f"Original rows : {original_rows:,}")
print(f"Distinct rows : {distinct_rows:,}")
print(f"Duplicates removed: {original_rows - distinct_rows:,}")

Original rows : 284,807
Distinct rows : 283,726
Duplicates removed: 1,081


In [ ]:
# ==========================
# Fraud Distribution
# ==========================

from pyspark.sql import Row
from pyspark.sql.functions import (
    broadcast,
    col,
    round
)

# Create lookup table
class_df = spark.createDataFrame([
    Row(Class=0, Description="Normal Transaction"),
    Row(Class=1, Description="Fraud")
])

# Calculate transactions by class
transactions_df = (
    df
    .groupBy("Class")
    .count()
    .withColumnRenamed("count", "Transactions")
)

# Calculate total number of transactions
total_transactions = df.count()

# Calculate percentage of transactions
transactions_df = (
    transactions_df
    .withColumn(
        "Percentage",
        round(
            col("Transactions") / total_transactions * 100,
            4
        )
    )
)

# Join lookup table with results
result = (
    broadcast(class_df)   
    .join(
        transactions_df,
        on="Class",
        how="inner"
    )
    .orderBy("Class")
)

# Display results
result.show()

+-----+------------------+------------+----------+
|Class|       Description|Transactions|Percentage|
+-----+------------------+------------+----------+
|    0|Normal Transaction|      284315|   99.8273|
|    1|             Fraud|         492|    0.1727|
+-----+------------------+------------+----------+

